In [2]:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

In [3]:
import duckdb
import altair as alt

%load_ext sql
conn = duckdb.connect()
%sql conn --alias duckdb

Tip: You may define configurations in /gpfs1/home/j/s/jstonge1/scisciDB/pyproject.toml or /users/j/s/jstonge1/.jupysql/config.

Did not find user configurations in /gpfs1/home/j/s/jstonge1/scisciDB/pyproject.toml.

In [5]:
%%sql
INSTALL ducklake;
LOAD ducklake;
ATTACH 'ducklake:metadata.ducklake' AS scisciDB (DATA_PATH '/gpfs1/home/j/s/jstonge1/data/scisciDB/');
USE scisciDB;

,Success


To lookup catalogs, we can inspect the `__ducklake_metadata_scisciDB.ducklake_*` tables. Usually, we can use `SHOW ALL TABLES;` to see them. But for some reason the jupyter extension in vsCODE doesn't display output tables. Here's the catalogs:

```
┌───────────────────────────────────────┐
│                 name                  │
│                varchar                │
├───────────────────────────────────────┤
│ ducklake_column                       │
│ ducklake_column_mapping               │
│ ducklake_column_tag                   │
│ ducklake_data_file                    │
│ ducklake_delete_file                  │
│ ducklake_file_column_stats            │
│ ducklake_file_partition_value         │
│ ducklake_files_scheduled_for_deletion │
│ ducklake_inlined_data_tables          │
│ ducklake_metadata                     │
│ ducklake_name_mapping                 │
│ ducklake_partition_column             │
│ ducklake_partition_info               │
│ ducklake_schema                       │
│ ducklake_schema_versions              │
│ ducklake_snapshot                     │
│ ducklake_snapshot_changes             │
│ ducklake_table                        │
│ ducklake_table_column_stats           │
│ ducklake_table_stats                  │
│ ducklake_tag                          │
│ ducklake_view                         │
├───────────────────────────────────────┤
│                22 rows                │
└───────────────────────────────────────┘
```

In [6]:
%%sql 
SELECT * FROM __ducklake_metadata_scisciDB.ducklake_metadata;

,key,value,scope,scope_id
0,version,0.3,None,<NA>
1,created_by,DuckDB 68d7555f68,None,<NA>
2,data_path,/gpfs1/home/j/s/jstonge1/data/scisciDB/,None,<NA>
3,encrypted,false,None,<NA>


In [7]:
%sql SELECT * FROM __ducklake_metadata_scisciDB.ducklake_table_stats;

,table_id,record_count,next_row_id,file_size_bytes
0,1,461047006,461047006,114852578385
1,2,11993991,11993991,294310036929


In [8]:
%sql SELECT * FROM __ducklake_metadata_scisciDB.ducklake_table;

,table_id,table_uuid,begin_snapshot,end_snapshot,schema_id,table_name,path,path_is_relative
0,1,019abcdb-ba13-7a03-a4ba-5dd337bbc797,1,<NA>,0,s2_papers,s2_papers/,True
1,2,019ac61f-6b3d-7aa9-9320-9370b9aa8a9f,3,<NA>,0,s2orc_v2,s2orc_v2/,True


In [9]:
%sql SELECT * FROM __ducklake_metadata_scisciDB.ducklake_snapshot;

,snapshot_id,snapshot_time,schema_version,next_catalog_id,next_file_id
0,0,2025-11-25 16:11:47.427021-05:00,0,1,0
1,1,2025-11-25 16:11:47.464952-05:00,1,2,113
2,2,2025-11-27 11:16:23.211588-05:00,2,2,113
3,3,2025-11-27 11:22:16.689942-05:00,3,3,190
4,4,2025-11-27 11:58:28.546571-05:00,4,3,190
5,5,2025-11-27 11:58:33.200277-05:00,5,3,190
6,6,2025-11-27 11:58:33.278482-05:00,5,3,339


## Inspect `s2_papers`

In [10]:
%%sql
select count(*) as total_papers from scisciDB.s2_papers;

,total_papers
0,230523503


In [11]:
%%sql 
count_df << SELECT 
    COUNT(*) as n, year 
    FROM scisciDB.s2_papers 
    WHERE 
        year IS NOT NULL AND year > 1900
    GROUP BY year;

In [12]:
alt.Chart(count_df).mark_bar().encode(
    x=alt.X('year:O', 
            axis=alt.Axis(labelAngle=-45, 
                         values=list(count_df['year'][::5]))),  # Show every 5th year
    y=alt.Y('n:Q', title='Count'),
    tooltip=['year:O', 'n:Q']
).properties(
    width=700,
    height=200
).configure_axis(
    grid=True,
    gridOpacity=0.3
)

alt.Chart(...)

This is pretty damn cool. I cannot overemphasize enough how it used to be more messy than that to do this count.

Just take a moment to realize what we just accomplished; in 2.3s, we analyze 270G worth of paper data and we only have 46G of RAM on the current node. We did that thanks to duckdb, but then with the `sql` magic in jupyter notebook we could produce an interactive visualization using `altair`. This is just amazing for exploring big-ish datasets.

Say that we want to know the proportion for which we have parsed text? Although this is conceptually simple question, it is technically made more challenging by the fact that we need to somehow know which papers has been parsed or not. The thing is that `s2_papers` don't have this field in the raw data, but we know which papers has been parsed by looking at the `s2orc_v2` table, containing the full-body parsed text. Usually, this is a simple join, where we look up `corpusid` on both tables; but here this join involves tow large tables; 270Gb and ?. 

But this is no problem for duckdb! We can just do something like without using that much RAM

In [13]:
# %%sql
# ALTER TABLE scisciDB.s2_papers ADD COLUMN has_fulltext BOOLEAN;
# UPDATE scisciDB.s2_papers AS p
# SET has_fulltext = EXISTS (
#     SELECT 1 FROM scisciDB.s2orc_v2 AS o
#     WHERE o.corpusid = p.corpusid
# );

In [14]:
%%sql
fulltext_df << SELECT 
    year,
    COUNT(*) as total_papers,
    COUNT(*) FILTER (WHERE has_fulltext) as papers_with_fulltext,
    COUNT(*) FILTER (WHERE NOT has_fulltext) as papers_without_fulltext
    FROM scisciDB.s2_papers 
    WHERE year IS NOT NULL AND year > 1900
    GROUP BY year
    ORDER BY year;

In [16]:
# Create base chart
base = alt.Chart(fulltext_df).encode(
    x=alt.X('year:Q',
            axis=alt.Axis(labelAngle=-45, values=list(fulltext_df['year'][::5])),
            title='Year')
)

# Bottom bar (total) - orange
bottom = base.mark_bar(color='orange', opacity=0.8).encode(
    y=alt.Y('total_papers:Q',
            scale=alt.Scale(type='symlog'),
            title='Number of Papers'),
    tooltip=['year', 'total_papers']
)

# Top bar (with fulltext) - blue
top = base.mark_bar(color='steelblue', opacity=0.8).encode(
    y=alt.Y('papers_with_fulltext:Q',
            scale=alt.Scale(type='symlog')),
    tooltip=['year', 'papers_with_fulltext']
)

(bottom + top).properties(
    width=800,
    height=200,
    title='Full Text Availability by Year'
).configure_axis(
    grid=True,
    gridOpacity=0.3
)

alt.LayerChart(...)